In [2]:
import os, zipfile
import numpy as np
import tensorflow as tf
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

from google.colab import drive
drive.mount("/content/drive")

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print("Extracted BASE folders:", os.listdir(BASE))


Mounted at /content/drive
Extracted BASE folders: ['New hand pd Dataset', 'Hand Pd']


In [6]:
!pip -q install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.4 MB/s eta 0:00:00


In [8]:
import imagehash
from PIL import Image
from collections import defaultdict

In [9]:


HANDPD_ROOT = os.path.join(BASE, "Hand Pd")
SPIRAL_ROOT = os.path.join(HANDPD_ROOT, "Spiral_HandPD")
MEANDER_ROOT = os.path.join(HANDPD_ROOT, "Meander_HandPD")

image_paths_hand = []
labels_hand = []

def collect_handpd(root):
    for cls in os.listdir(root):
        cls_path = os.path.join(root, cls)
        if not os.path.isdir(cls_path):
            continue

        cls_lower = cls.lower()
        if cls_lower == "healthy":
            lab = 0
        elif cls_lower == "parkinson":
            lab = 1
        else:
            continue

        for r, d, files in os.walk(cls_path):
            for f in files:
                if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                    image_paths_hand.append(os.path.join(r, f))
                    labels_hand.append(lab)

collect_handpd(SPIRAL_ROOT)
collect_handpd(MEANDER_ROOT)

print("HandPD total:", len(image_paths_hand))
print("Healthy:", labels_hand.count(0))
print("Parkinson:", labels_hand.count(1))


hash_to_paths = defaultdict(list)

for p in tqdm(image_paths_hand, desc="Hashing HandPD"):
    h = get_phash(p)
    if h is not None:
        hash_to_paths[str(h)].append(p)

hash_list = list(hash_to_paths.keys())
used = set()
final_keep_hashes = []

for i in tqdm(range(len(hash_list)), desc="Removing near duplicates HandPD"):
    if hash_list[i] in used:
        continue

    base_hash = imagehash.hex_to_hash(hash_list[i])
    used.add(hash_list[i])
    final_keep_hashes.append(hash_list[i])

    for j in range(i+1, len(hash_list)):
        if hash_list[j] in used:
            continue
        compare_hash = imagehash.hex_to_hash(hash_list[j])
        if base_hash - compare_hash <= PHASH_DISTANCE_THRESHOLD:
            used.add(hash_list[j])

path_to_label_hand = dict(zip(image_paths_hand, labels_hand))

clean_paths_handpd = []
clean_labels_handpd = []

for h in final_keep_hashes:
    p = hash_to_paths[h][0]
    clean_paths_handpd.append(p)
    clean_labels_handpd.append(path_to_label_hand[p])

print("\nCLEAN HANDPD")
print("Images:", len(clean_paths_handpd))
print("Healthy:", clean_labels_handpd.count(0))
print("Parkinson:", clean_labels_handpd.count(1))


HandPD total: 808
Healthy: 216
Parkinson: 592


Hashing HandPD:   0%|          | 0/808 [00:00<?, ?it/s]


NameError: name 'get_phash' is not defined